# Scorer Feature Analysis

`extract_scorer_features.py` が出力した parquet（layer1-4 の GAP 特徴 + メタ）を読み、
解析ロジックは `real_fonts_vs_gen_fonts.analysis.feature_analysis` の共通関数を呼ぶ。
図は同じ関数で出力ディレクトリに保存し、ここではインライン表示する
（CLI の `analyze_scorer_features.py` と完全に同じ実装）。

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from real_fonts_vs_gen_fonts.analysis import (
    load_features,
    fit_combined_embedding,
    plot_combined_embedding,
    plot_embedding_thumbnails,
    fit_linear_probe,
    layer_probe_auroc,
    per_group_stats,
    per_char_auroc,
    plot_group_stats,
    plot_score_distributions,
    plot_score_ranked_images,
    plot_misclassified_images,
    plot_group_representative_images,
    plot_worst_auroc_char_examples,
    has_images,
    set_font_scale,
    set_legend_loc,
)

# === 設定 ===
PARQUET_PATH = Path("outputs/<exp>/features/scorer_features.parquet")  # 要変更
OUT_DIR = PARQUET_PATH.parent.parent / "feature_analysis_nb"
IMAGE_ROOT = PARQUET_PATH.parent  # 入力サムネ (inputs/) は parquet の隣
REDUCER = "umap"          # "pca" | "tsne" | "umap"
EMBED_LAYER = "layer4"
EMBED_SAMPLES = 5000       # t-SNE/UMAP の fit 上限（-1 で全件）
STATS_SPLIT = "test"
FONT_SCALE = 2.0          # 図のフォントサイズ倍率（キャプション・タイトル・軸・凡例を一括拡縮）
LEGEND_LOC = "best"       # 凡例位置。"outside"=軸の外（被らないがプロットが小さくなる）/ matplotlib loc（内側）
OUT_DIR.mkdir(parents=True, exist_ok=True)
set_font_scale(FONT_SCALE)
set_legend_loc(LEGEND_LOC)

In [ ]:
df_meta, feats = load_features(PARQUET_PATH)
print(f"{len(df_meta):,} samples | layers={ {k: v.shape for k, v in feats.items()} }")
df_meta.head()

## 1. 同一空間プロット（全 split 結合で一括 fit）

In [ ]:
emb, idx = fit_combined_embedding(feats[EMBED_LAYER], reducer=REDUCER, max_samples=EMBED_SAMPLES)
embed_dir = OUT_DIR / "embedding"
plot_combined_embedding(emb, df_meta.iloc[idx], embed_dir, REDUCER)
for name in ("by_label_split", "by_score", "by_char"):
    display(Image(filename=str(embed_dir / f"embed_{REDUCER}_{name}.png")))

## 2. レイヤ別線形プローブ AUROC（train で fit → 各 split に適用）

In [ ]:
labels = df_meta["label"].to_numpy()
split_col = df_meta["split"].to_numpy()
tr = split_col == "train"
rows = []
for layer, X in feats.items():
    scaler, clf = fit_linear_probe(X[tr], labels[tr])
    for s in ["train", "val", "test"]:
        m = split_col == s
        if not m.any():
            continue
        _, _, auroc = layer_probe_auroc(scaler, clf, X[m], labels[m])
        rows.append({"layer": layer, "split": s, "auroc": auroc})
pd.DataFrame(rows).pivot(index="layer", columns="split", values="auroc")

## 3. 文字別統計（per-char AUROC は real/gen 両クラスがあるので定義可能）

In [ ]:
char_stats = per_group_stats(df_meta, "char")
char_auroc = per_char_auroc(df_meta)
stats_dir = OUT_DIR / "stats"
plot_group_stats(char_stats, "char", stats_dir, split=STATS_SPLIT, auroc=char_auroc)
display(Image(filename=str(stats_dir / f"per_char_score_{STATS_SPLIT}.png")))
display(Image(filename=str(stats_dir / f"per_char_auroc_{STATS_SPLIT}.png")))
char_auroc.sort_values("auroc")

## 4. フォント別統計（実フォントのみ。label=1 単一クラスのため AUROC は不可、スコア統計のみ）

In [ ]:
font_stats = per_group_stats(df_meta, "font_family")
plot_group_stats(font_stats, "font_family", stats_dir, split=STATS_SPLIT)
display(Image(filename=str(stats_dir / f"per_font_real_score_{STATS_SPLIT}.png")))
# 平均スコアが低い実フォント = モデルが「偽物っぽい」と判断するフォント
real_font = font_stats[(font_stats["split"] == STATS_SPLIT) & (font_stats["label"] == "real")]
real_font.sort_values("mean").head(10)

## 5. スコア / ロジット分布（real vs gen）

In [ ]:
dist_dir = OUT_DIR / "distributions"
plot_score_distributions(df_meta, dist_dir, split=STATS_SPLIT)
for col in ("score", "logit"):
    p = dist_dir / f"{col}_dist_{STATS_SPLIT}.png"
    if p.exists():
        display(Image(filename=str(p)))

## 5b. スコア KDE（real/gen 密度・平均・裾・混同領域 min(p_real,p_gen)）

`score_separation_stats` で OVL（重なり係数 ∫min）・混同点 s*・各平均を数値でも確認する。

In [ ]:
from real_fonts_vs_gen_fonts.analysis import plot_score_kde, score_separation_stats

plot_score_kde(df_meta, dist_dir, split=STATS_SPLIT)
p = dist_dir / f"score_kde_{STATS_SPLIT}.png"
if p.exists():
    display(Image(filename=str(p)))
print(score_separation_stats(df_meta, STATS_SPLIT))

## 6. 画像グリッド（モデル入力サムネ。`inputs/` がある parquet のみ）

各プロットは**単一カテゴリの N×N グリッド**を帯ごとに 1 枚ずつ出力する（設定はファイル名に記入）。
スコア順（top/middle/bottom）・誤判定（real/gen）・文字別/フォント別代表（lowest/middle/highest）・
AUROC 最悪文字（real/gen）。`SHOW_LABELS=False` で純画像グリッドになる。
`--no-save-inputs` で抽出した parquet には `inputs/` が無いのでスキップされる。

In [ ]:
GRID_N = 8           # 単一カテゴリ N×N グリッドの N（8 = 64 枚）
SHOW_LABELS = True   # False で各セルの score/char ラベルを消し純画像グリッドに

if not has_images(df_meta, IMAGE_ROOT):
    print("No input thumbnails (image_path) in this parquet — image grids skipped.")
else:
    img_dir = OUT_DIR / "images"
    plot_embedding_thumbnails(emb, df_meta.iloc[idx], IMAGE_ROOT, OUT_DIR / "embedding", REDUCER)
    plot_score_ranked_images(df_meta, IMAGE_ROOT, img_dir, split=STATS_SPLIT, grid_n=GRID_N, show_labels=SHOW_LABELS)
    plot_misclassified_images(df_meta, IMAGE_ROOT, img_dir, split=STATS_SPLIT, grid_n=GRID_N, show_labels=SHOW_LABELS)
    plot_group_representative_images(df_meta, IMAGE_ROOT, img_dir, "char", split=STATS_SPLIT, grid_n=GRID_N, show_labels=SHOW_LABELS)
    plot_group_representative_images(df_meta, IMAGE_ROOT, img_dir, "font_family", split=STATS_SPLIT, grid_n=GRID_N, show_labels=SHOW_LABELS)
    plot_worst_auroc_char_examples(df_meta, char_auroc, IMAGE_ROOT, img_dir, split=STATS_SPLIT, grid_n=GRID_N, show_labels=SHOW_LABELS)
    # each plot writes one single-category N×N grid per band; the selection is in the filename
    for name in [
        f"embedding/embed_{REDUCER}_thumbnails.png",
        f"images/ranked_real_top_{STATS_SPLIT}.png",
        f"images/ranked_gen_bottom_{STATS_SPLIT}.png",
        f"images/misclassified_real_{STATS_SPLIT}.png",
        f"images/misclassified_gen_{STATS_SPLIT}.png",
        f"images/repr_char_lowest_{STATS_SPLIT}.png",
        f"images/repr_font_family_lowest_{STATS_SPLIT}.png",
        f"images/worst_auroc_real_{STATS_SPLIT}.png",
        f"images/worst_auroc_gen_{STATS_SPLIT}.png",
    ]:
        p = OUT_DIR / name
        if p.exists():
            display(Image(filename=str(p)))

## 7. KDE 領域の代表画像（real/gen × 平均・裾・混同点 s*）

領域ごとに単一カテゴリ N×N グリッドを出力（`kde_{real,gen}_{mean,tail,overlap}_<split>.png`）。
`real_overlap` と `gen_overlap` は同じスコア帯（混同点 s* 近傍）なので、2 枚を並べて real と gen を
直接見比べられる（＝スコアラーが見分けられないグリフ）。

In [ ]:
from real_fonts_vs_gen_fonts.analysis import plot_kde_region_images

if not has_images(df_meta, IMAGE_ROOT):
    print("No input thumbnails (image_path) in this parquet — KDE region images skipped.")
else:
    plot_kde_region_images(df_meta, IMAGE_ROOT, OUT_DIR / "images", split=STATS_SPLIT, grid_n=GRID_N, show_labels=SHOW_LABELS)
    # real_overlap / gen_overlap cover the same score band (the confusion point s*)
    for name in ["real_mean", "gen_mean", "real_overlap", "gen_overlap"]:
        p = OUT_DIR / "images" / f"kde_{name}_{STATS_SPLIT}.png"
        if p.exists():
            display(Image(filename=str(p)))